# SAGIP-AI: train the local photo classifier (all in one notebook)

This one notebook does the whole job, top to bottom:

1. builds the training set (the photos already in Drive, plus every photo listed in `training/extra-photos.tsv`, plus your own photos)
2. removes duplicates and any photo that is also in the benchmark
3. trains the model (8 incident types, optional severity head)
4. tests it on a locked test set, and on the 50-photo benchmark if you uploaded it
5. exports it to ONNX for the app's `local` provider
6. saves everything to `MyDrive/sagip/runs/<timestamp>/`, including `metrics.json`

**How to run:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
Allow the Drive connection when asked. No API key or secret is needed: all photo sources are public.

**What it reads from Google Drive** (all under `MyDrive/sagip/`):

| Path | Needed? | What it is |
|---|---|---|
| `dataset.zip` | yes | The training set from the last run (`dataset/<type>/...`). Already in Drive |
| `benchmark.zip` | optional | Your `benchmark/dataset` folder zipped (with `labels.csv`). Gives the benchmark score and missed urgent, and keeps those photos out of training |
| `own-photos/<type>/` | optional | Your own phone photos, one folder per incident type |
| `severity.csv` | optional | Severity labels for training photos (`file,severity`, paths like `fire/abc.jpg`), made with `npm run label-severity` and checked by the team. Turns on the severity head (ISS-016) |

## 1. Settings
The defaults are the best run so far (2026-10-25: 88% type accuracy on the benchmark). Change only if you know why.

In [ ]:
DRIVE_ROOT   = '/content/drive/MyDrive/sagip'
REPO_BRANCH  = 'dev'          # branch of github.com/helidastar/SAGIP-AI to read the photo list and app constants from

DOWNLOAD_EXTRA_PHOTOS = True  # download photos in training/extra-photos.tsv that are not in dataset.zip yet
MAX_SIDE     = 512            # longest side of downloaded photos (training uses 224 px)

MODEL_NAME   = 'efficientnet_b2'  # keep it small: it must fit in a Vercel function (R-017). Not convnext_tiny.
IMG_SIZE     = 224
BATCH_SIZE   = 32
HEAD_EPOCHS  = 5     # step 1: train only the new final layers
FULL_EPOCHS  = 15    # step 2: fine-tune the whole model
SEED         = 42
REVIEW_THRESHOLD = 0.70  # CONFIDENCE_THRESHOLD in src/lib/constants.ts: below this a report goes to human review
URGENT_RECALL_TARGET = 0.95  # severity head: share of truly high/critical validation photos it must call high/critical (sets the urgent cutoff)

QUICK_TEST   = False  # True = a few minutes on a small sample, only to check the notebook works. Never use its numbers.

## 2. Connect Drive, check the GPU, get the code

In [ ]:
import os, sys, time, json, glob, shutil, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # running outside Colab (local test): DRIVE_ROOT must already exist
WORK = '/content' if IN_COLAB else os.environ.get('SAGIP_WORK', '/tmp/sagip-work')
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(DRIVE_ROOT):
    raise FileNotFoundError(f'{DRIVE_ROOT} not found. Check that Drive is connected and the sagip folder exists in My Drive.')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
if device == 'cpu' and not QUICK_TEST:
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all again.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'onnx', 'onnxscript', 'onnxruntime', 'scikit-learn'], check=True)

REPO = os.path.join(WORK, 'SAGIP-AI')
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', REPO_BRANCH, 'https://github.com/helidastar/SAGIP-AI.git', REPO], check=True)
REPO_COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Code:', REPO_BRANCH, REPO_COMMIT)

RUN_ID = time.strftime('%Y%m%d-%H%M%S')
RUN_DIR = os.path.join(DRIVE_ROOT, 'runs', RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
STARTED = time.time()
print('Results will be saved to', RUN_DIR)

In [ ]:
# The app's own lists, read from the code so the notebook can never drift from the app.
import re
const_ts = open(os.path.join(REPO, 'src/lib/constants.ts')).read()
EXPECTED_CLASSES = re.findall(r'"(\w+)"', re.search(r'INCIDENT_TYPES = \[(.*?)\]', const_ts, re.S).group(1))
SEVERITY_CLASSES = ['low', 'moderate', 'high', 'critical']
sev_rank = dict(re.findall(r'(\w+): (\d)', re.search(r'export const SEVERITY = \{(.*?)\}', const_ts).group(1)))
assert [s for s, _ in sorted(sev_rank.items(), key=lambda kv: kv[1])] == SEVERITY_CLASSES, sev_rank
assert len(EXPECTED_CLASSES) == 8, EXPECTED_CLASSES

# Keyword severity (src/lib/ai/keyword.ts): what the app uses for severity when the model has no severity head.
kw_ts = open(os.path.join(REPO, 'src/lib/ai/keyword.ts')).read()
sev_block = kw_ts[kw_ts.index('SEVERITY_KEYWORDS'):kw_ts.index('DEFAULT_SEVERITY')]
SEVERITY_KEYWORDS = [(level, re.findall(r'"([^"]+)"', words))
                     for level, words in re.findall(r'\["(\w+)", \[(.*?)\]\]', sev_block, re.S)]
def_block = kw_ts[kw_ts.index('const DEFAULT_SEVERITY'):]
DEFAULT_SEVERITY = dict(re.findall(r'(\w+): "(\w+)"', def_block[:def_block.index('};')]))
assert [l for l, _ in SEVERITY_KEYWORDS] == ['critical', 'high', 'moderate'] and set(DEFAULT_SEVERITY) == set(EXPECTED_CLASSES)
print('Incident types:', EXPECTED_CLASSES)

## 3. Build the training set
Starts from `dataset.zip`, then adds the new photos from `extra-photos.tsv` and `own-photos/`.
Same rules as `npm run dataset:prepare`: skip unreadable or tiny photos (under 128 px), skip duplicates
(perceptual hash, 4 of 64 bits), skip a photo filed under two types, and never train on a benchmark photo.

In [ ]:
import zipfile, csv, io, collections, concurrent.futures as cf, urllib.request, unicodedata
import numpy as np
from PIL import Image, ImageFile, ImageOps
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp', '.tif', '.tiff'}
MIN_SIDE, NEAR_DUP_BITS = 128, 4

def find_zip_root(extract_dir, must_have):
    """Folder inside the extracted zip that holds `must_have` (the zip may have an extra top folder)."""
    for root, dirs, files in os.walk(extract_dir):
        if any(m in dirs or m in files for m in must_have):
            return root
    return None

def list_images(folder):
    out = []
    for root, _, files in os.walk(folder):
        out += [os.path.join(root, f) for f in files if os.path.splitext(f)[1].lower() in IMAGE_EXT]
    return sorted(out)

def dhash(path):
    """64-bit difference hash (same idea as prepare-dataset.ts); None if unreadable or too small."""
    try:
        with Image.open(path) as im:
            if min(im.size) < MIN_SIDE: return None
            im.draft('L', (64, 64))
            px = np.asarray(ImageOps.exif_transpose(im).convert('L').resize((9, 8), Image.BILINEAR), dtype=np.int16)
        bits = (px[:, :-1] > px[:, 1:]).flatten()
        return int(''.join('1' if b else '0' for b in bits), 2)
    except Exception:
        return None

POP8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
class HashIndex:
    """Finds a near-duplicate (<= NEAR_DUP_BITS different bits) among the hashes added so far."""
    def __init__(self): self.h = np.zeros(0, np.uint64); self.labels = []
    def add(self, h, label): self.h = np.append(self.h, np.uint64(h)); self.labels.append(label)
    def match(self, h):
        if not len(self.h): return None
        dist = POP8[(self.h ^ np.uint64(h)).view(np.uint8)].reshape(-1, 8).sum(1)
        i = int(dist.argmin())
        return self.labels[i] if dist[i] <= NEAR_DUP_BITS else None

def hash_all(paths):
    with cf.ThreadPoolExecutor(8) as ex: return list(ex.map(dhash, paths))

# 3a. Base set from dataset.zip
DATA_DIR = os.path.join(WORK, 'dataset')
shutil.rmtree(DATA_DIR, ignore_errors=True)
base_zip = os.path.join(DRIVE_ROOT, 'dataset.zip')
if not os.path.exists(base_zip):
    raise FileNotFoundError(f'{base_zip} not found. Upload the training set zip (dataset/<type>/...) to MyDrive/sagip/.')
print(f'Unzipping {base_zip} ({os.path.getsize(base_zip) / 1e6:.0f} MB) ...')
tmp = os.path.join(WORK, 'unzip-base'); shutil.rmtree(tmp, ignore_errors=True)
with zipfile.ZipFile(base_zip) as z: z.extractall(tmp)
root = find_zip_root(tmp, EXPECTED_CLASSES)
if not root: raise FileNotFoundError('dataset.zip has no folders named after the incident types: ' + ', '.join(EXPECTED_CLASSES))
shutil.move(root, DATA_DIR); shutil.rmtree(tmp, ignore_errors=True)
unknown = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and d not in EXPECTED_CLASSES]
if unknown: raise ValueError(f'Unknown folders in dataset.zip (fix the names): {unknown}')

# 3b. Benchmark photos: kept out of training, used for the benchmark in step 8
BENCH_DIR, bench_rows = None, []
bench_zip = os.path.join(DRIVE_ROOT, 'benchmark.zip')
if os.path.exists(bench_zip):
    tmp = os.path.join(WORK, 'benchmark'); shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(bench_zip) as z: z.extractall(tmp)
    BENCH_DIR = find_zip_root(tmp, ['labels.csv'])
    if BENCH_DIR:
        with open(os.path.join(BENCH_DIR, 'labels.csv'), newline='', encoding='utf-8-sig') as f:
            bench_rows = [r for r in csv.DictReader(f) if r.get('file')]
        print('Benchmark:', len(bench_rows), 'photos in labels.csv')
    else:
        print('WARNING: benchmark.zip has no labels.csv, so the benchmark step is skipped.')
else:
    print('No benchmark.zip in Drive: the benchmark step (score, missed urgent) is skipped.')
bench_index = HashIndex()
for h in hash_all(list_images(BENCH_DIR) if BENCH_DIR else []):
    if h is not None: bench_index.add(h, 'benchmark')

skipped = collections.Counter()
kept = HashIndex()
base_files = list_images(DATA_DIR)
for path, h in zip(base_files, hash_all(base_files)):
    t = os.path.relpath(path, DATA_DIR).split(os.sep)[0]
    if h is None: os.remove(path); skipped['unreadable_or_small'] += 1; continue
    if bench_index.match(h): os.remove(path); skipped['in_benchmark'] += 1; continue
    kept.add(h, t)
n_base = len(kept.labels)
print('Photos from dataset.zip:', n_base)

def add_photo(src_path, type_, dest_rel, h):
    """Adds one new photo unless it is unreadable, a duplicate, a benchmark photo or filed under another type."""
    if h is None: skipped['unreadable_or_small'] += 1; return False
    if bench_index.match(h): skipped['in_benchmark'] += 1; return False
    other = kept.match(h)
    if other is not None:
        skipped['duplicate' if other == type_ else 'filed_under_two_types'] += 1; return False
    dest = os.path.join(DATA_DIR, type_, dest_rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with Image.open(src_path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB'); im.thumbnail((MAX_SIDE, MAX_SIDE))
        im.save(dest, 'JPEG', quality=85)
    kept.add(h, type_); return True

# 3c. New photos from training/extra-photos.tsv (public Roboflow Universe photos, CC BY 4.0, no key needed)
extra = collections.Counter(); extra_failed = 0
if DOWNLOAD_EXTRA_PHOTOS:
    rows = [l.rstrip('\n').split('\t') for l in open(os.path.join(REPO, 'training/extra-photos.tsv'), encoding='utf-8') if l.strip()]
    def photo_key(p):  # '<type>/<file name without extension>'
        rel = os.path.relpath(p, DATA_DIR).split(os.sep)
        return rel[0] + '/' + os.path.splitext(rel[-1])[0]
    have = {photo_key(p) for p in list_images(DATA_DIR)}
    # dataset.zip stores these photos as <type>/<id>.jpg, so a photo already there is not downloaded again.
    todo = [r for r in rows if f'{r[0]}/{r[2]}' not in have]
    print(f'extra-photos.tsv: {len(rows)} photos, {len(rows) - len(todo)} already in dataset.zip, {len(todo)} to download')
    if QUICK_TEST: todo = todo[:40]
    print(f'Downloading {len(todo)} photos ...')
    dl_dir = os.path.join(WORK, 'downloads'); os.makedirs(dl_dir, exist_ok=True)
    def fetch(r):
        type_, source, id_, url = r[:4]
        out = os.path.join(dl_dir, f'{type_}__{source}__{id_}.img')
        if not os.path.exists(out):
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'sagip-ai-training'})
                with urllib.request.urlopen(req, timeout=30) as resp: data = resp.read()
                with open(out, 'wb') as f: f.write(data)
            except Exception:
                return r, None
        return r, out
    with cf.ThreadPoolExecutor(16) as ex: results = list(ex.map(fetch, todo))
    ok = [(r, p) for r, p in results if p]
    extra_failed = len(results) - len(ok)
    for (r, p), h in zip(ok, hash_all([p for _, p in ok])):
        type_, source, id_ = r[0], r[1], r[2]
        if type_ not in EXPECTED_CLASSES: skipped['unknown_type'] += 1; continue
        if add_photo(p, type_, os.path.join(f'roboflow-{source}', f'{id_}.jpg'), h): extra[type_] += 1
    print('Added from extra-photos.tsv:', dict(extra), f'({extra_failed} downloads failed)' if extra_failed else '')

# 3d. Your own photos from MyDrive/sagip/own-photos/<type>/
own = collections.Counter()
own_dir = os.path.join(DRIVE_ROOT, 'own-photos')
if os.path.isdir(own_dir):
    bad = [d for d in os.listdir(own_dir) if os.path.isdir(os.path.join(own_dir, d)) and d not in EXPECTED_CLASSES]
    if bad: print('WARNING: own-photos folders with unknown names are skipped:', bad)
    for t in EXPECTED_CLASSES:
        files = list_images(os.path.join(own_dir, t))
        for i, (p, h) in enumerate(zip(files, hash_all(files))):
            name = re.sub(r'[^\w-]+', '_', os.path.splitext(os.path.basename(p))[0])[:60]
            if add_photo(p, t, os.path.join('own', f'{name}-{i}.jpg'), h): own[t] += 1
    print('Added own photos:', dict(own))
else:
    print('No own-photos folder in Drive (optional).')

if QUICK_TEST:  # keep a small, balanced sample
    for t in EXPECTED_CLASSES:
        for p in list_images(os.path.join(DATA_DIR, t))[40:]: os.remove(p)

# 3e. Severity labels (optional): MyDrive/sagip/severity.csv overrides one inside dataset.zip
sev_drive = os.path.join(DRIVE_ROOT, 'severity.csv')
if os.path.exists(sev_drive):
    shutil.copy(sev_drive, os.path.join(DATA_DIR, 'severity.csv'))
    print('Using severity labels from', sev_drive)

DATASET_INFO = {
    'from_dataset_zip': n_base, 'added_extra_photos': dict(extra), 'extra_download_failed': extra_failed,
    'added_own_photos': dict(own), 'skipped': dict(skipped),
}
print('Skipped:', dict(skipped))

## 4. Check the dataset

In [ ]:
import random
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import timm
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

full = datasets.ImageFolder(DATA_DIR)
classes = full.classes  # alphabetical order = the model's output order (saved in labels.json)
missing = sorted(set(EXPECTED_CLASSES) - set(classes))
if missing: print('WARNING: no photos for', missing, '- the model will never predict these')
counts = collections.Counter(full.targets)
for i, c in enumerate(classes):
    print(f'{c:20s} {counts[i]:5d}' + ('  <- too few, aim for 100+' if counts[i] < 100 else ''))
print('Total:', len(full))
DATASET_INFO['per_type'] = {c: counts[i] for i, c in enumerate(classes)}
DATASET_INFO['total'] = len(full)

# Severity head (ISS-016, docs/local-severity-design.md): optional per-photo labels in dataset/severity.csv
# (columns file,severity, paths relative to dataset/). Photos with no label are skipped in the severity loss only.
severity_by_path = {}
sev_csv = os.path.join(DATA_DIR, 'severity.csv')
if os.path.exists(sev_csv):
    with open(sev_csv, newline='') as f:
        for row in csv.DictReader(f):
            if row.get('severity') in SEVERITY_CLASSES:
                p = os.path.join(DATA_DIR, row['file'].replace('\\', '/'))
                if os.path.exists(p): severity_by_path[p] = SEVERITY_CLASSES.index(row['severity'])
HAS_SEVERITY = len(severity_by_path) > 0
if HAS_SEVERITY:
    sev_counts = collections.Counter(severity_by_path.values())
    print(f'\nSeverity labels: {len(severity_by_path)} of {len(full)} photos')
    for i, s in enumerate(SEVERITY_CLASSES):
        print(f'  {s:10s} {sev_counts.get(i, 0):5d}' + ('  <- aim for 150+' if sev_counts.get(i, 0) < 150 else ''))
else:
    print('\nNo severity labels: training type only. The app keeps using keyword severity.')
DATASET_INFO['severity_labels'] = len(severity_by_path)

## 5. Split 70 / 15 / 15 (stratified)
The **test set is locked**: never tune on it.

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(full)); y = np.array(full.targets)
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=y[tmp_idx], random_state=SEED)
print(len(tr_idx), 'train /', len(va_idx), 'val /', len(te_idx), 'test')
with open(os.path.join(RUN_DIR, 'test_files.json'), 'w') as f:
    json.dump([os.path.relpath(full.samples[i][0], DATA_DIR) for i in te_idx], f, indent=1)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.3),                         # low light, bad phone cameras
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),   # motion blur, rain
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([   # same as src/lib/ai/local.ts preprocess()
    transforms.Resize(int(IMG_SIZE * 256 / 224)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class Subset(Dataset):
    def __init__(self, indices, tf): self.indices, self.tf = indices, tf
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, label = full.samples[self.indices[i]]
        return self.tf(Image.open(path).convert('RGB')), label, severity_by_path.get(path, -1)  # -1 = no severity label

workers = 2 if IN_COLAB else 0
train_dl = DataLoader(Subset(tr_idx, train_tf), BATCH_SIZE, shuffle=True, num_workers=workers, pin_memory=device == 'cuda')
val_dl   = DataLoader(Subset(va_idx, eval_tf), BATCH_SIZE, num_workers=workers)
test_dl  = DataLoader(Subset(te_idx, eval_tf), BATCH_SIZE, num_workers=workers)

## 6. Train
One backbone with two small heads: incident type (every photo) and severity (only photos with a severity label).

In [ ]:
class DualHead(nn.Module):
    def __init__(self, backbone_name, num_types, num_severities, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.type_head = nn.Linear(self.backbone.num_features, num_types)
        self.severity_head = nn.Linear(self.backbone.num_features, num_severities)
    def forward(self, x):
        feats = self.backbone(x)
        return self.type_head(feats), self.severity_head(feats)

model = DualHead(MODEL_NAME, len(classes), len(SEVERITY_CLASSES)).to(device)

# Class weights so rare types are not ignored.
tr_counts = np.bincount(y[tr_idx], minlength=len(classes)).astype(float)
weights = torch.tensor(tr_counts.sum() / (len(classes) * np.maximum(tr_counts, 1)), dtype=torch.float32, device=device)
type_loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
# Severity tiers get class weights too, so the rarer tiers (often critical) are not ignored.
sev_tr = np.bincount([severity_by_path[full.samples[i][0]] for i in tr_idx if full.samples[i][0] in severity_by_path],
                     minlength=len(SEVERITY_CLASSES)).astype(float)
sev_weights = torch.tensor(sev_tr.sum() / (len(SEVERITY_CLASSES) * np.maximum(sev_tr, 1)), dtype=torch.float32, device=device) if sev_tr.sum() else None
sev_loss_fn = nn.CrossEntropyLoss(weight=sev_weights)
SEVERITY_WEIGHT = 0.5  # loss = type loss + 0.5 x severity loss (docs/local-severity-design.md)

def run_epoch(dl, opt=None):
    model.train(opt is not None)
    total = correct = sev_total = sev_correct = 0; loss_sum = 0.0
    with torch.set_grad_enabled(opt is not None):
        for xb, yb, sb in dl:
            xb, yb, sb = xb.to(device), yb.to(device), sb.to(device)
            type_out, sev_out = model(xb)
            loss = type_loss_fn(type_out, yb)
            m = sb >= 0
            if m.any():
                loss = loss + SEVERITY_WEIGHT * sev_loss_fn(sev_out[m], sb[m])
                sev_correct += (sev_out[m].argmax(1) == sb[m]).sum().item(); sev_total += int(m.sum().item())
            if opt: opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(yb); total += len(yb)
            correct += (type_out.argmax(1) == yb).sum().item()
    return loss_sum / total, correct / total, (sev_correct / sev_total if sev_total else float('nan'))

best_acc, best_path, history = -1.0, os.path.join(RUN_DIR, 'best.pt'), []
def train(epochs, lr, stage):
    global best_acc
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(1, epochs + 1):
        tl, ta, _ = run_epoch(train_dl, opt); vl, va, vsa = run_epoch(val_dl); sched.step()
        mark = ''
        if va > best_acc: best_acc = va; torch.save(model.state_dict(), best_path); mark = '  * saved'
        history.append({'stage': stage, 'epoch': ep, 'train_acc': round(ta, 4), 'val_acc': round(va, 4), 'val_loss': round(vl, 4)})
        sev = f'  val severity acc {vsa:.3f}' if vsa == vsa else ''
        print(f'[{stage}] epoch {ep:2d}  train acc {ta:.3f}  val acc {va:.3f}  val loss {vl:.3f}{sev}{mark}')

head_epochs, full_epochs = (1, 1) if QUICK_TEST else (HEAD_EPOCHS, FULL_EPOCHS)
TRAIN_STARTED = time.time()
# Step 1: freeze the pretrained layers, train only the new heads
for p in model.parameters(): p.requires_grad = False
for p in list(model.type_head.parameters()) + list(model.severity_head.parameters()): p.requires_grad = True
train(head_epochs, 1e-3, 'head')
# Step 2: unfreeze everything, fine-tune at a low learning rate
for p in model.parameters(): p.requires_grad = True
train(full_epochs, 1e-4, 'full')
TRAIN_MINUTES = round((time.time() - TRAIN_STARTED) / 60, 1)
print('Best val accuracy:', round(best_acc, 3), f'({TRAIN_MINUTES} min)')

## 7. Test on the locked test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
model.load_state_dict(torch.load(best_path, map_location=device)); model.eval()
probs, labels, sev_probs, sev_labels = [], [], [], []
with torch.no_grad():
    for xb, yb, sb in test_dl:
        type_out, sev_out = model(xb.to(device))
        probs.append(torch.softmax(type_out, 1).cpu()); labels.append(yb)
        m = sb >= 0
        if m.any(): sev_probs.append(torch.softmax(sev_out, 1).cpu()[m]); sev_labels.append(sb[m])
probs = torch.cat(probs).numpy(); labels = torch.cat(labels).numpy()
pred, conf = probs.argmax(1), probs.max(1)
test_acc = float((pred == labels).mean())
brier = float(np.mean((conf - (pred == labels)) ** 2))
auto = conf >= REVIEW_THRESHOLD
print(f'Test accuracy: {test_acc:.3f}  ({len(labels)} photos)\n')
print(classification_report(labels, pred, labels=range(len(classes)), target_names=classes, zero_division=0))
report = classification_report(labels, pred, labels=range(len(classes)), target_names=classes, zero_division=0, output_dict=True)

print('confidence    n   accuracy   (calibration: is "90% sure" right about 90% of the time?)')
bins = np.linspace(0, 1, 6)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & ((conf < hi) if hi < 1 else (conf <= hi))
    if m.any(): print(f'{lo:.1f}-{hi:.1f}  {m.sum():5d}   {(pred[m] == labels[m]).mean():.3f}')
print(f'\nAt threshold {REVIEW_THRESHOLD}: {auto.mean():.1%} auto-classified, accuracy on those '
      f'{(pred[auto] == labels[auto]).mean() if auto.any() else 0:.3f}.  Brier: {brier:.3f} (lower is better)')

TEST = {
    'n': int(len(labels)), 'type_accuracy': round(test_acc, 4), 'brier': round(brier, 4),
    'auto_classified_rate': round(float(auto.mean()), 4),
    'accuracy_when_auto_classified': round(float((pred[auto] == labels[auto]).mean()), 4) if auto.any() else None,
    'per_type': {c: {'precision': round(report[c]['precision'], 3), 'recall': round(report[c]['recall'], 3),
                     'f1': round(report[c]['f1-score'], 3), 'n': int(report[c]['support'])} for c in classes},
    'confusion_matrix': {'rows_expected_cols_predicted': classes,
                         'values': confusion_matrix(labels, pred, labels=range(len(classes))).tolist()},
}
if sev_labels:
    sp = torch.cat(sev_probs).numpy(); sl = torch.cat(sev_labels).numpy(); spred = sp.argmax(1)
    TEST['severity'] = {'n': int(len(sl)), 'exact': round(float((spred == sl).mean()), 4),
                        'within_one': round(float((np.abs(spred - sl) <= 1).mean()), 4),
                        'brier': round(float(np.mean((sp.max(1) - (spred == sl)) ** 2)), 4)}
    print('Severity on the labeled test photos:', TEST['severity'])

cm = np.array(TEST['confusion_matrix']['values'])
fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right'); ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Expected'); ax.set_title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, 'confusion_matrix.png'), dpi=150); plt.show()

## 7b. Severity calibration (only with a severity head)
Two settings are picked on the **validation** photos (never the test or benchmark photos) and saved in `labels.json`,
where `src/lib/ai/severity.ts` uses them:

- `severity_temperature`: scales the severity scores so that "80% sure" is right about 80% of the time.
- `severity_urgent_cutoff`: if P(high) + P(critical) reaches it, the model answers high or critical. It is the highest cutoff
  that still catches `URGENT_RECALL_TARGET` of the truly high/critical validation photos. A missed urgent report costs more than an extra review.

The app also sends a report to review when the severity confidence is below 0.70 (`src/lib/ai/review.ts`).


In [ ]:
RANK = {s: i for i, s in enumerate(SEVERITY_CLASSES)}
HI, CR = RANK['high'], RANK['critical']

def pick_severity(scores, temperature=1.0, urgent_cutoff=None):
    """Same rule as src/lib/ai/severity.ts. Returns (tier index, confidence)."""
    z = np.asarray(scores, dtype=np.float64) / (temperature if temperature > 0 else 1.0)
    p = np.exp(z - z.max()); p /= p.sum()
    top = int(p.argmax())
    if urgent_cutoff is not None and p[HI] + p[CR] >= urgent_cutoff: top = CR if p[CR] > p[HI] else HI
    return top, float(p[top])

def severity_logits(dl):
    out, lab = [], []
    with torch.no_grad():
        for xb, _, sb in dl:
            m = sb >= 0
            if m.any(): out.append(model(xb.to(device))[1].cpu()[m]); lab.append(sb[m])
    return (torch.cat(out).numpy(), torch.cat(lab).numpy()) if out else (np.zeros((0, 4)), np.zeros(0, int))

def severity_stats(logits, labels, t, cut):
    picks = [pick_severity(z, t, cut) for z in logits]
    pred = np.array([k for k, _ in picks]); conf = np.array([c for _, c in picks])
    urgent = labels >= HI
    return {'n': int(len(labels)), 'exact': round(float((pred == labels).mean()), 4),
            'within_one': round(float((np.abs(pred - labels) <= 1).mean()), 4),
            'urgent_recall': round(float((pred[urgent] >= HI).mean()), 4) if urgent.any() else None,
            'urgent_n': int(urgent.sum()),
            'review_rate_from_severity': round(float(((pred >= HI) | (conf < REVIEW_THRESHOLD)).mean()), 4),
            'brier': round(float(np.mean((conf - (pred == labels)) ** 2)), 4)}

SEV_CAL = {'skipped': 'no severity labels'}
SEV_T, SEV_CUT = 1.0, None
if HAS_SEVERITY:
    model.load_state_dict(torch.load(best_path, map_location=device)); model.eval().to(device)
    val_z, val_y = severity_logits(val_dl)
    if len(val_y) < 30:
        print(f'Only {len(val_y)} labeled validation photos: too few to calibrate, using temperature 1 and no cutoff.')
    else:
        def nll(t):
            z = val_z / t; z = z - z.max(1, keepdims=True)
            return float(-(z[np.arange(len(val_y)), val_y] - np.log(np.exp(z).sum(1))).mean())
        SEV_T = float(min(np.arange(0.5, 5.01, 0.05), key=nll))
        urgent = val_y >= HI
        SEV_CUT = 0.05
        for cut in np.arange(0.95, 0.04, -0.05):
            preds = np.array([pick_severity(z, SEV_T, cut)[0] for z in val_z[urgent]])
            if urgent.any() and (preds >= HI).mean() >= URGENT_RECALL_TARGET: SEV_CUT = round(float(cut), 2); break
    test_z, test_y = severity_logits(test_dl)
    SEV_CAL = {'temperature': round(SEV_T, 2), 'urgent_cutoff': SEV_CUT, 'urgent_recall_target': URGENT_RECALL_TARGET,
               'val': severity_stats(val_z, val_y, SEV_T, SEV_CUT) if len(val_y) else None,
               'test_plain_argmax': severity_stats(test_z, test_y, 1.0, None) if len(test_y) else None,
               'test_calibrated': severity_stats(test_z, test_y, SEV_T, SEV_CUT) if len(test_y) else None,
               'note': 'Labels are Gemini drafts checked by the team, not ground truth: these numbers measure agreement with those labels.'}
    print(json.dumps(SEV_CAL, indent=2))
else:
    print('No severity head: skipped.')


## 8. Export to ONNX
Makes the three files the app's `local` provider needs: `sagip-classifier.onnx`, `sagip-classifier.onnx.data`
(if the exporter splits the weights out) and `labels.json`. The severity output is only exported when the
severity head was actually trained, because `src/lib/ai/local.ts` trusts that output whenever it exists.

In [ ]:
class TypeOnly(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)[0]

model.eval().cpu()
export_model = (model if HAS_SEVERITY else TypeOnly(model)).eval()
outputs = ['logits', 'severity'] if HAS_SEVERITY else ['logits']
for f in glob.glob(os.path.join(RUN_DIR, 'sagip-classifier.onnx*')): os.remove(f)
onnx_path = os.path.join(RUN_DIR, 'sagip-classifier.onnx')
torch.onnx.export(export_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), onnx_path,
                  input_names=['image'], output_names=outputs,
                  dynamic_axes={'image': {0: 'batch'}, **{o: {0: 'batch'} for o in outputs}}, opset_version=18)
labels_json = {'classes': classes, 'image_size': IMG_SIZE, 'mean': MEAN, 'std': STD, 'model': MODEL_NAME}
if HAS_SEVERITY:
    labels_json['severity_classes'] = SEVERITY_CLASSES
    labels_json['severity_temperature'] = round(SEV_T, 2)
    if SEV_CUT is not None: labels_json['severity_urgent_cutoff'] = SEV_CUT
json.dump(labels_json, open(os.path.join(RUN_DIR, 'labels.json'), 'w'), indent=2)
MODEL_FILES = sorted(os.path.basename(f) for f in glob.glob(os.path.join(RUN_DIR, 'sagip-classifier.onnx*'))) + ['labels.json']
size_mb = sum(os.path.getsize(os.path.join(RUN_DIR, f)) for f in MODEL_FILES) / 1e6
print('Model files:', MODEL_FILES, f'({size_mb:.1f} MB)')

# Check the ONNX file gives the same answers as PyTorch
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
xb, _, _ = next(iter(test_dl))
with torch.no_grad(): torch_out = export_model(xb)
torch_type = (torch_out[0] if HAS_SEVERITY else torch_out).numpy()
onnx_type = sess.run(None, {'image': xb.numpy()})[0]
ONNX_MAX_DIFF = float(np.abs(torch_type - onnx_type).max())
ONNX_SAME = bool((torch_type.argmax(1) == onnx_type.argmax(1)).all())
print('Max difference:', ONNX_MAX_DIFF, '(should be < 0.001)   Same predictions:', ONNX_SAME)
assert ONNX_SAME, 'ONNX export gives different answers from PyTorch: do not use these files.'

## 9. Benchmark (only if `benchmark.zip` is in Drive)
Runs the exported ONNX file on the benchmark photos the same way the app does (`src/lib/ai/local.ts`):
type from the photo, severity from the severity head if there is one, otherwise from the description
keywords (`src/lib/ai/keyword.ts`). Metrics and score follow `scripts/benchmark.ts` (Appendix B.4).
The official number is still `npm run benchmark -- --models local --yes` on the laptop; this is the same method run here.

In [ ]:
def normalize(text):
    text = unicodedata.normalize('NFKD', (text or '').lower())
    text = ''.join(ch if unicodedata.category(ch)[0] in 'LN' or ch == '-' or ch.isspace() else ' ' for ch in text)
    return ' ' + re.sub(r'\s+', ' ', text).strip() + ' '

def severity_from_text(description, incident_type):
    text = normalize(description)
    for level, words in SEVERITY_KEYWORDS:
        if any(f' {w} ' in text for w in words): return level
    return DEFAULT_SEVERITY[incident_type]

def preprocess(path):  # same steps as local.ts: rotate by EXIF, resize shorter side to 256, center-crop 224
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        return eval_tf(im).unsqueeze(0).numpy()

def softmax(v): e = np.exp(v - v.max()); return e / e.sum()

BENCH = {'skipped': 'no benchmark.zip in MyDrive/sagip'}
if bench_rows:
    rank = {s: i for i, s in enumerate(SEVERITY_CLASSES)}
    runs, errors = [], []
    for r in bench_rows:
        p = os.path.join(BENCH_DIR, r['file'])
        if not os.path.exists(p) or r.get('incidentType') not in EXPECTED_CLASSES or r.get('severity') not in rank:
            errors.append(r['file']); continue
        t0 = time.time(); out = sess.run(None, {'image': preprocess(p)}); ms = (time.time() - t0) * 1000
        pt = softmax(out[0][0]); best = int(pt.argmax()); itype = classes[best]
        if HAS_SEVERITY:
            k, sev_conf = pick_severity(out[1][0], SEV_T, SEV_CUT); sev = SEVERITY_CLASSES[k]
        else:
            sev, sev_conf = severity_from_text(r.get('description', ''), itype), None
        runs.append({'file': r['file'], 'expected_type': r['incidentType'], 'expected_severity': r['severity'],
                     'type': itype, 'severity': sev, 'confidence': float(pt[best]), 'severity_confidence': sev_conf, 'ms': ms})
    if errors: print('WARNING: skipped benchmark rows (missing photo or bad label):', errors)
    n = len(runs)
    type_ok = [x['type'] == x['expected_type'] for x in runs]
    sev_diff = [abs(rank[x['severity']] - rank[x['expected_severity']]) for x in runs]
    needs_review = [x['confidence'] < REVIEW_THRESHOLD or (x['severity_confidence'] is not None and x['severity_confidence'] < REVIEW_THRESHOLD)
                    or x['severity'] in ('high', 'critical') for x in runs]  # review.ts (no hoax check in the image model)
    missed = [x['file'] for x, rv in zip(runs, needs_review)
              if rank[x['expected_severity']] >= rank['high'] and rank[x['severity']] < rank['high'] and not rv]
    b_type, b_exact = np.mean(type_ok), np.mean([d == 0 for d in sev_diff])
    b_within, b_brier = np.mean([d <= 1 for d in sev_diff]), np.mean([(x['confidence'] - ok) ** 2 for x, ok in zip(runs, type_ok)])
    success = n / (n + len(errors))
    # Appendix B.4 weights. Cost ($0) and latency score 1.0: the local model is the cheapest and fastest in every comparison run.
    score = round(100 * (0.35 * b_type * success + 0.25 * b_within * success + 0.15 * (1 - b_brier) * success + 0.15 + 0.10), 1)
    confusion = collections.defaultdict(collections.Counter)
    for x in runs: confusion[x['expected_type']][x['type']] += 1
    BENCH = {
        'n': n, 'errors': len(errors), 'score': score, 'type_accuracy': round(float(b_type), 4),
        'severity_exact': round(float(b_exact), 4), 'severity_within_one': round(float(b_within), 4),
        'brier': round(float(b_brier), 4), 'review_rate': round(float(np.mean(needs_review)), 4),
        'missed_urgent': len(missed), 'missed_urgent_files': missed,
        'severity_from': 'model severity head' if HAS_SEVERITY else 'description keywords (keyword.ts)',
        'mean_ms_per_photo_colab_cpu': round(float(np.mean([x['ms'] for x in runs])), 1),
        'wrong_type': [f"{x['file']}: {x['expected_type']} -> {x['type']} ({x['confidence']:.2f})" for x, ok in zip(runs, type_ok) if not ok],
        'confusion': {k: dict(v) for k, v in confusion.items()},
        'method': 'ONNX on Colab CPU, local.ts preprocessing; metrics and weights from scripts/benchmark.ts',
    }
    with open(os.path.join(RUN_DIR, 'benchmark_runs.json'), 'w') as f: json.dump(runs, f, indent=1)
    print(f"Benchmark ({n} photos): score {score}  type {b_type:.1%}  severity ±1 {b_within:.1%}  "
          f"missed urgent {len(missed)}  review rate {np.mean(needs_review):.1%}")
    for w in BENCH['wrong_type']: print('  wrong:', w)
else:
    print('Benchmark skipped:', BENCH['skipped'])

## 10. Save `metrics.json`
This is the file to send back. It is also printed below.

In [ ]:
METRICS = {
    'run': RUN_ID, 'code': f'{REPO_BRANCH}@{REPO_COMMIT}', 'quick_test': QUICK_TEST,
    'settings': {'model': MODEL_NAME, 'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'head_epochs': head_epochs,
                 'full_epochs': full_epochs, 'seed': SEED, 'review_threshold': REVIEW_THRESHOLD},
    'dataset': DATASET_INFO,
    'split': {'train': len(tr_idx), 'val': len(va_idx), 'test': len(te_idx)},
    'best_val_accuracy': round(best_acc, 4), 'train_minutes': TRAIN_MINUTES, 'history': history,
    'test': TEST,
    'severity_calibration': SEV_CAL,
    'benchmark': BENCH,
    'onnx': {'files': MODEL_FILES, 'size_mb': round(size_mb, 1), 'has_severity_output': HAS_SEVERITY,
             'max_diff_vs_pytorch': ONNX_MAX_DIFF, 'same_predictions': ONNX_SAME},
    'total_minutes': round((time.time() - STARTED) / 60, 1),
}
with open(os.path.join(RUN_DIR, 'metrics.json'), 'w') as f: json.dump(METRICS, f, indent=2)
print('Saved to', RUN_DIR, '\n')
print(json.dumps({k: v for k, v in METRICS.items() if k != 'history'}, indent=2))

## Done
Everything is in `MyDrive/sagip/runs/<timestamp>/`:
- `metrics.json`: dataset counts, test results, benchmark score and missed urgent, ONNX check
- `sagip-classifier.onnx`, `sagip-classifier.onnx.data` (if made) and `labels.json`: copy **all of them together** into `models/`
  (an old `.onnx.data` next to a new `.onnx` gives nonsense)
- `confusion_matrix.png`, `test_files.json`, `benchmark_runs.json` (if the benchmark ran) and `best.pt`

Before using a new model in the app, compare it with the current one on the benchmark (`npm run benchmark -- --models local --yes`).
If test accuracy drops, look at the confusion matrix and add photos to the types that get mixed up.